In [216]:
import pandas as pd
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.linear_model import LinearRegression, RidgeCV, LassoCV
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

In [217]:
data = pd.read_csv('salary.csv', sep=',')
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 52 entries, 0 to 51
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   sex          52 non-null     str  
 1   rank         52 non-null     str  
 2   year         52 non-null     int64
 3   degree       52 non-null     str  
 4   year_degree  52 non-null     int64
 5   salary       52 non-null     int64
dtypes: int64(3), str(3)
memory usage: 2.6 KB


In [218]:
for col in ['sex', 'rank', 'degree']:
    data[col] = data[col].astype('category')
data.describe(include='all')

,sex,rank,year,degree,year_degree,salary
count,52,52,52.000000,52,52.000000,52.000000
unique,2,3,NaN,2,NaN,NaN
top,male,full,NaN,doctorate,NaN,NaN
freq,38,20,NaN,34,NaN,NaN
mean,NaN,NaN,7.480769,NaN,16.115385,23797.653846
std,NaN,NaN,5.507536,NaN,10.222340,5917.289154
min,NaN,NaN,0.000000,NaN,1.000000,15000.000000
25%,NaN,NaN,3.000000,NaN,6.750000,18246.750000
50%,NaN,NaN,7.000000,NaN,15.500000,23719.000000
75%,NaN,NaN,11.000000,NaN,23.250000,27258.500000


In [219]:
data['sex'].unique()

['male', 'female']
Categories (2, str): ['female', 'male']

In [220]:
data['rank'].unique()

['full', 'associate', 'assistant']
Categories (3, str): ['assistant', 'associate', 'full']

In [221]:
data['degree'].unique()

['doctorate', 'masters']
Categories (2, str): ['doctorate', 'masters']

In [222]:
sex_categories = ['male', 'female']
degree_categories = ['masters', 'doctorate']

binary_encoder = OneHotEncoder(
    categories=[sex_categories, degree_categories],
    sparse_output=False,
    drop='if_binary',
    handle_unknown='ignore' 
)
encoded_array = binary_encoder.fit_transform(data[['sex', 'degree']])
encoded_df = pd.DataFrame(encoded_array, columns=['sex_encoded', 'degree_encoded'])
data_encoded = pd.concat([data, encoded_df], axis=1)
data_encoded.head(10)


,sex,rank,year,degree,year_degree,salary,sex_encoded,degree_encoded
0,male,full,25,doctorate,35,36350,0.0,1.0
1,male,full,13,doctorate,22,35350,0.0,1.0
2,male,full,10,doctorate,23,28200,0.0,1.0
3,female,full,7,doctorate,27,26775,1.0,1.0
4,male,full,19,masters,30,33696,0.0,0.0
5,male,full,16,doctorate,21,28516,0.0,1.0
6,female,full,0,masters,32,24900,1.0,0.0
7,male,full,16,doctorate,18,31909,0.0,1.0
8,male,full,13,masters,30,31850,0.0,0.0
9,male,full,13,masters,31,32850,0.0,0.0


In [223]:
rank_encoder = OrdinalEncoder(
    categories=[['assistant', 'associate', 'full']],
    handle_unknown="use_encoded_value",
    unknown_value=-1
)

data_encoded['rank_encoded'] = rank_encoder.fit_transform(data[['rank']])
data_encoded.groupby("rank", as_index=False).first()

,rank,sex,year,degree,year_degree,salary,sex_encoded,degree_encoded,rank_encoded
0,assistant,male,16,masters,23,19175,0.0,0.0,0.0
1,associate,male,15,doctorate,19,24750,0.0,1.0,1.0
2,full,male,25,doctorate,35,36350,0.0,1.0,2.0


In [ ]:
data_encoded.drop(columns=['rank', 'sex', 'degree' ], inplace=True)
y = data_encoded['salary']
X = data_encoded.drop(columns=['salary'

In [225]:
def get_linear_regression_coefficients(lr, feature_names):
    coef = pd.DataFrame(lr.coef_.ravel(), index=feature_names, columns=["coef"])
    coef.loc["intercept"] = lr.intercept_
    return coef

In [226]:
lr = LinearRegression()
lr.fit(X_train, y_train)
y_pred = lr.predict(X_test)
mean_squared_error(y_test, y_pred)

8604203.53348391

In [227]:
get_linear_regression_coefficients(lr, X_train.columns)

,coef
year,524.234211
year_degree,-203.142232
sex_encoded,2281.824808
degree_encoded,-1572.561801
rank_encoded,6451.619509
intercept,17150.332533


In [ ]:
lr_ridge = RidgeCV(alphas=[100])
lr_ridge.fit(X_train, y_train)
y_pred_ridge = lr_ridge.predict(X_test)
mean_squared_error(y_test, y_pred_ridge)

3657774.5805429076

In [242]:
get_linear_regression_coefficients(lr_ridge, X_train.columns)

,coef
year,426.716791
year_degree,49.021443
sex_encoded,328.265848
degree_encoded,365.243154
rank_encoded,3379.777513
intercept,16162.277697


In [230]:
lr_lasso = LassoCV(alphas=[1e-3, 1e-2, 1e-1, 1])
lr_lasso.fit(X_train, y_train)
y_pred_lasso = lr_lasso.predict(X_test)
mean_squared_error(y_test, y_pred_lasso)

8581322.371818816

In [231]:
get_linear_regression_coefficients(lr_lasso, X_train.columns)

,coef
year,523.372858
year_degree,-201.916076
sex_encoded,2270.068867
degree_encoded,-1556.803140
rank_encoded,6442.264201
intercept,17138.717117
